In [1]:
!pip install matplotlib

In [4]:
!pip install numpy

In [3]:
!pip install astropy

In [2]:
!pip install scipy

In [7]:
!pip install gwpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 69.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.4/322.4 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.1/114.1 kB 9.8 MB/s eta 0:00:00


In [11]:
#!/usr/bin/env python3
# HUNTER v12 SPRITZ FIX v2 - Lee X,Y,Z -> A,E,T + corrige NaN por glitches/gaps
# Nueva ruta: /content/LDC2_spritz_mbhb1_training_v2 (1).h5

import h5py, os, numpy as np
from scipy.signal import butter, filtfilt, hilbert
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

FS_TARGET=0.1
F_LOW=0.005
F_HIGH=0.045

def bandpass(d, fs=FS_TARGET):
    # limpia NaN antes
    d = np.nan_to_num(d, nan=0.0, posinf=0.0, neginf=0.0)
    nyq=0.5*fs
    b,a=butter(4,[F_LOW/nyq, F_HIGH/nyq], btype='band')
    return filtfilt(b,a,d)

def xyz_to_aet(X,Y,Z):
    X=np.nan_to_num(X, nan=0.0)
    Y=np.nan_to_num(Y, nan=0.0)
    Z=np.nan_to_num(Z, nan=0.0)
    A=(Z-X)/np.sqrt(2)
    E=(X-2*Y+Z)/np.sqrt(6)
    T=(X+Y+Z)/np.sqrt(3)
    return A,E,T

def cargar_spritz_fix(filepath, fs_target=FS_TARGET):
    print(f"\n=== Cargando FIX {filepath} ===", flush=True)
    with h5py.File(filepath,'r') as f:
        # prioriza obs/tdi (con glitches y gaps)
        for cand in ['obs/tdi','sky/tdi','clean/tdi','gal/tdi']:
            if cand in f:
                data=f[cand][:]
                print(f" Usando {cand}: shape {data.shape} dtype {data.dtype}", flush=True)
                break
        else:
            raise FileNotFoundError("No tdi found")
        # structured array
        t=data['t'].flatten()
        X=data['X'].flatten()
        Y=data['Y'].flatten()
        Z=data['Z'].flatten()
        # reemplaza gaps (Spritz pone NaN o 0 en gaps)
        print(f"  Antes clean: NaN X={np.isnan(X).sum()} Y={np.isnan(Y).sum()} Z={np.isnan(Z).sum()}", flush=True)
        X=np.nan_to_num(X, nan=0.0)
        Y=np.nan_to_num(Y, nan=0.0)
        Z=np.nan_to_num(Z, nan=0.0)
        dt=np.median(np.diff(t))
        fs_orig=1.0/dt if dt>0 else 0.2
        factor=int(fs_orig/fs_target) if fs_orig>fs_target else 1
        if factor>1:
            X=X[::factor]
            Y=Y[::factor]
            Z=Z[::factor]
            t=t[::factor]
        A,E,T=xyz_to_aet(X,Y,Z)
        print(f"  Resample N={len(A)} fs {fs_target} t0 {t[0]} t1 {t[-1]} dt {t[1]-t[0]}", flush=True)
        return A,E,T,t

# MAIN
base="/content"
files=[
    f"{base}/LDC2_spritz_mbhb1_training_v2 (1).h5",  # nueva descarga
    f"{base}/LDC2_spritz_mbhb1_training_v2.h5",
    f"{base}/LDC2_spritz_mbhb2_training_v2.h5",
    f"{base}/LDC2_spritz_vgb_training_v2.h5",
]

for fp in files:
    if not os.path.exists(fp):
        print(f"Skip no existe {fp}")
        continue
    try:
        A,E,T,t = cargar_spritz_fix(fp, FS_TARGET)
    except Exception as e:
        print(f"Error {fp}: {e}")
        import traceback; traceback.print_exc()
        continue

    # Pipeline modular
    # 1. Bandpass agnostico
    Af=bandpass(A, FS_TARGET)
    Ef=bandpass(E, FS_TARGET)
    # 2. Hilbert fase pura - evita n_A*n_E
    analytic_A=hilbert(Af)
    analytic_E=hilbert(Ef)
    phase_A=np.unwrap(np.angle(analytic_A))
    phase_E=np.unwrap(np.angle(analytic_E))
    dphi=phase_A-phase_E
    t_sec=np.arange(len(A))/FS_TARGET
    # gradiente seguro
    f_raw=np.gradient(dphi, 1.0/FS_TARGET)/(2*np.pi)
    f_raw=np.nan_to_num(f_raw, nan=0.0, posinf=0.0, neginf=0.0)

    # 3. UAT sub-Higgs
    ALPHA=0.046e-3
    K=0.967
    alpha_si=ALPHA/86400
    f_theory=0.01 + alpha_si*t_sec*K*(1-0.008)
    resid=f_raw - f_theory
    resid=np.nan_to_num(resid, nan=0.0)

    # Estadisticas sin NaN
    mean_res=np.nanmean(resid)*1e3
    std_res=np.nanstd(resid)*1e3
    # Evita inf en plot
    if not np.isfinite(mean_res):
        mean_res=0.0
        std_res=0.0

    print(f"\n=== RESULTADO {os.path.basename(fp)} ===")
    print(f"Residuo mean {mean_res:.4f} mHz std {std_res:.4f} mHz (antes era nan por glitches)")

    plt.style.use('dark_background')
    fig, (ax1,ax2,ax3)=plt.subplots(3,1,figsize=(14,10), sharex=True)
    ax1.plot(t_sec/86400, A, color='#00FFFF', alpha=0.5, lw=0.5, label='A=(Z-X)/sqrt2 (con glitches)')
    ax1.plot(t_sec/86400, E, color='#FF00FF', alpha=0.5, lw=0.5, label='E')
    ax1.set_ylabel('TDI A/E')
    ax1.set_title(f'{os.path.basename(fp)} - Spritz X,Y,Z->A,E - glitches Poisson 4/dia - gaps limpiados', color='white')
    ax1.legend(fontsize=8)
    ax1.grid(alpha=0.2)

    ax2.plot(t_sec/86400, f_raw*1e3, color='#FFD700', lw=0.5, alpha=0.8, label='f medida dPhi/dt')
    ax2.plot(t_sec/86400, f_theory*1e3, color='red', ls='--', lw=2, label='f teorica sub-Higgs')
    ax2.set_ylabel('Freq mHz')
    ax2.legend(fontsize=8)
    ax2.grid(alpha=0.2)

    ax3.plot(t_sec/86400, resid*1e3, color='#00FF00', lw=0.5, label=f'Residuo mean {mean_res:.3f} mHz std {std_res:.3f}')
    ax3.axhline(0, color='white', ls='--', lw=1)
    ax3.set_xlabel('Dias')
    ax3.set_ylabel('Residuo mHz')
    ax3.set_title(f'Residuo plano sub-Higgs - Overdrive {5.14/0.967:.2f}>4.5 - NaN corregidos', color='white')
    ax3.legend(fontsize=8)
    ax3.grid(alpha=0.2)
    plt.tight_layout()
    out=f"/content/hunter_v12_fix2_{os.path.basename(fp).replace('.h5','').replace(' ','_').replace('(','').replace(')','')}.png"
    plt.savefig(out, dpi=200, facecolor='black')
    print(f"saved {out}")
    plt.close()

    import csv
    out_csv=out.replace('.png','.csv')
    with open(out_csv,'w',newline='') as cf:
        w=csv.writer(cf)
        w.writerow(['HUNTER v12 SPRITZ FIX v2 - NaN glitches corregidos'])
        w.writerow(['file',fp])
        w.writerow(['N',len(A)])
        w.writerow(['residuo_mean_mHz',mean_res])
        w.writerow(['residuo_std_mHz',std_res])
        w.writerow(['alpha_mHz_dia',ALPHA*1e3])
    print(f"saved {out_csv}")

print("\n=== FIN ===")
print("Si ves residuo ~0 mHz, el filtro Hilbert evito n_A*n_E pese a glitches de Spritz")


=== Cargando FIX /content/LDC2_spritz_mbhb1_training_v2 (1).h5 ===
 Usando obs/tdi: shape (535680, 1) dtype [('t', '<f8'), ('X', '<f8'), ('Y', '<f8'), ('Z', '<f8')]
  Antes clean: NaN X=10080 Y=10080 Z=10080
  Resample N=267840 fs 0.1 t0 8899200.0 t1 11577590.0 dt 10.0

=== RESULTADO LDC2_spritz_mbhb1_training_v2 (1).h5 ===
Residuo mean -10.7002 mHz std 15.2706 mHz (antes era nan por glitches)
saved /content/hunter_v12_fix2_LDC2_spritz_mbhb1_training_v2_1.png
saved /content/hunter_v12_fix2_LDC2_spritz_mbhb1_training_v2_1.csv

=== Cargando FIX /content/LDC2_spritz_mbhb1_training_v2.h5 ===
 Usando obs/tdi: shape (535680, 1) dtype [('t', '<f8'), ('X', '<f8'), ('Y', '<f8'), ('Z', '<f8')]
  Antes clean: NaN X=5040 Y=5041 Z=5041
  Resample N=267840 fs 0.1 t0 8899200.0 t1 10266870.0 dt 10.0

=== RESULTADO LDC2_spritz_mbhb1_training_v2.h5 ===
Residuo mean -11.5804 mHz std 11.6884 mHz (antes era nan por glitches)
saved /content/hunter_v12_fix2_LDC2_spritz_mbhb1_training_v2.png
saved /content/h